# Metadata Overview

Exploratory notebook for `data/metadata/grouped_df.csv`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import os

repo_root = Path.cwd().resolve().parents[1]
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

metadata_path = src_path / "../" / "data/metadata/grouped_df.csv"
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Metadata file not found: {metadata_path}")
df = pd.read_csv(metadata_path)

print(f"Loaded: {metadata_path}")
print(f"Shape: {df.shape}")
print()
print("Columns:")
for column in df.columns:
    print(f"- {column}")

Loaded: /home/csantiago/inescgarcia/diffusion-based-counterfactual-generation/src/../data/metadata/grouped_df.csv
Shape: (20000, 16)

Columns:
- image_id
- patient_id
- laterality
- view
- finding_categories
- finding_birads
- breast_birads
- breast_density
- split
- fold
- Mass
- Suspicious_Calcification
- resized_xmin
- resized_ymin
- resized_xmax
- resized_ymax


In [2]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "n_unique": [df[column].nunique(dropna=False) for column in df.columns],
    "n_null": df.isna().sum().values,
})
display(overview)

,column,dtype,n_unique,n_null
0,image_id,str,20000,0
1,patient_id,str,5000,0
2,laterality,str,2,0
3,view,str,2,0
4,finding_categories,str,153,0
5,finding_birads,str,74,0
6,breast_birads,str,5,0
7,breast_density,str,4,0
8,split,str,2,0
9,fold,int64,4,0


In [3]:
print("\nUnique values for columns with n_unique < 10:")
for column in df.columns:
    n_unique = df[column].nunique(dropna=False)
    if n_unique < 10:
        print(f"\n{column}")
        print(f"  {sorted(df[column].unique().tolist())}")


Unique values for columns with n_unique < 10:

laterality
  ['L', 'R']

view
  ['CC', 'MLO']

breast_birads
  ['BI-RADS 1', 'BI-RADS 2', 'BI-RADS 3', 'BI-RADS 4', 'BI-RADS 5']

breast_density
  ['DENSITY A', 'DENSITY B', 'DENSITY C', 'DENSITY D']

split
  ['test', 'training']

fold
  [0, 1, 2, 3]

Mass
  [0, 1]

Suspicious_Calcification
  [0, 1]


In [4]:
print("\nSample values for columns with n_unique > 10:")
for column in df.columns:
    n_unique = df[column].nunique(dropna=False)
    if n_unique > 10:
        sample_values = df[column].drop_duplicates().head(3).tolist()
        print(f"\n{column}")
        for _, val in enumerate(sample_values, 1):
            print(f"  {val}")


Sample values for columns with n_unique > 10:

image_id
  000470cbf12fe2b285cba99286a9a4fa.png
  000611f8c6a44659a1813f4019241829.png
  00095c0bc0043119471c227b056939e5.png

patient_id
  85189e7e7c1feac81ad1ed803679d5b6
  6f87ecb7d9ca57a790c3f81b23940af9
  d955eb69b20823839ef6ceb6c2b5c8e9

finding_categories
  ["['No Finding']"]
  ["['Mass']", "['Suspicious Calcification']", "['Suspicious Calcification']"]
  ["['Suspicious Calcification']"]

finding_birads
  [0]
  ['BI-RADS 4', 'BI-RADS 4', 'BI-RADS 4']
  ['BI-RADS 4']

resized_xmin
  [0.0]
  [360.32677, 411.5235916, 539.8412347]
  [444.3935178]

resized_ymin
  [0.0]
  [893.1984573, 937.212087, 930.1114816]
  [820.6548225]

resized_xmax
  [0.0]
  [726.6313048, 467.4569232, 630.86999]
  [524.3224904]

resized_ymax
  [0.0]
  [1114.277635, 982.4784465, 1011.768444]
  [902.0513535]


In [5]:
print("Split distribution:")
display(df["split"].value_counts(dropna=False).rename_axis("split").to_frame("count"))

print("Fold distribution:")
display(df["fold"].value_counts(dropna=False).sort_index().rename_axis("fold").to_frame("count"))

print("BI-RADS distribution per split:")
breast_birads_by_split = pd.crosstab(df["split"], df["breast_birads"])
display(breast_birads_by_split)
display((breast_birads_by_split.div(breast_birads_by_split.sum(axis=1), axis=0) * 100).round(2))

Split distribution:


,count
split,
training,16000
test,4000


Fold distribution:


,count
fold,
0,5000
1,5000
2,5000
3,5000


BI-RADS distribution per split:


breast_birads,BI-RADS 1,BI-RADS 2,BI-RADS 3,BI-RADS 4,BI-RADS 5
split,,,,,
test,2682,934,186,152,46
training,10724,3742,744,610,180


breast_birads,BI-RADS 1,BI-RADS 2,BI-RADS 3,BI-RADS 4,BI-RADS 5
split,,,,,
test,67.05,23.35,4.65,3.80,1.15
training,67.03,23.39,4.65,3.81,1.12


In [6]:
print("Patient/fold consistency check:")
if "patient_id" not in df.columns or "fold" not in df.columns:
    raise ValueError("Missing required columns: patient_id and/or fold")

patient_fold_counts = df.groupby("patient_id", dropna=False)["fold"].nunique(dropna=False)
violations = patient_fold_counts[patient_fold_counts > 1]

if violations.empty:
    print("No patients appear in more than one fold.")
else:
    print(f"Found {len(violations)} patients in more than one fold.")
    display(violations.sort_values(ascending=False).rename("n_folds").to_frame())

print("Mass and Suspicious_Calcification by split:")
display(df.groupby("split")[ ["Mass", "Suspicious_Calcification"] ].sum())

print("Mass and Suspicious_Calcification by fold:")
display(df.groupby("fold")[ ["Mass", "Suspicious_Calcification"] ].sum())

Patient/fold consistency check:
No patients appear in more than one fold.
Mass and Suspicious_Calcification by split:


,Mass,Suspicious_Calcification
split,,
test,219,105
training,894,337


Mass and Suspicious_Calcification by fold:


,Mass,Suspicious_Calcification
fold,,
0,272,134
1,276,99
2,287,81
3,278,128
